<a href="https://colab.research.google.com/github/expaetra/CM_final_project/blob/master/01_category_volume.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To ensure that there are enough training examples for each of the classes, arXiv categories were first evaluated based on how many papers they have. Using the arXiv API the total number of papers in each category was retrieved. Categories with fewer than 5000 papers were excluded, since there was a clear drop in volume separating well-represented categories from smaller ones.

For the other categories, equal number of papers is sampled from each. This prevented larger categories from dominating the dataset and kept the training data balanced across classes.

In [3]:
# import libraries

import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
import time
import pandas as pd

In [4]:
# all CS categories to be evaluated form https://arxiv.org/category_taxonomy

CATEGORIES = [
    "cs.AI", # artificial intelligence
    "cs.AR", # hardware architecture
    "cs.CC", # computational complexity
    "cs.CE", # computational engineering, finance, science
    "cs.CG", # computational geometry
    "cs.CL", # computation and language
    "cs.CR", # cryptography and security
    "cs.CV", # computer vision and pattern recognition
    "cs.CY", # computer vision and pattern recognition
    "cs.DB", # databases
    "cs.DC", # distributed, parallel and cluster computing
    "cs.DL", # digital libraries
    "cs.DM", # discrete mathematics
    "cs.DS", # data structures and algorithms
    "cs.ET", # emerging technologies
    "cs.FL", # formal languages, automata
    "cs.GL", # general
    "cs.GT", # computer science and game theory
    "cs.HC", # human-computer interaction
    "cs.IR", # information retrieval
    "cs.IT", # information theory
    "cs.LO", # logic
    "cs.LG", # machine learning
    "cs.MA", # multiagent systems
    "cs.MS", # mathematical software
    "cs.MM", # multimedia
    "cs.NE", # neural and evolutionary computing
    "cs.NI", # networking and internet architecture
    "cs.OH", # other
    "cs.OS", # operating systems
    "cs.PF", # performance
    "cs.PL", # programming languages
    "cs.RO", # robotics
    "cs.SC", # signal processing
    "cs.SD", # sound
    "cs.SE", # software engineering
    "cs.SI", # social and information networks
]

In [5]:
BASE_URL = "http://export.arxiv.org/api/query"
NS ={"opensearch": "http://a9.com/-/spec/opensearch/1.1/"}

Query the arXiv API for a category and return the total number of papers in that category. Request only 1 result since metadata is only counted - not the papers themselves.

In [6]:
# function to return total number of papers

def get_cat_volume(category):
  # build query apram for the api request
  params = urllib.parse.urlencode({
      "search_query":f"cat:{category}",
      "max_results":1, # reutrn only metadata (not the papers)
  })
  url = f"{BASE_URL}?{params}"

  try:
    with urllib.request.urlopen(url) as response:
      xml_data = response.read().decode("utf-8")

      # parse the response
      root = ET.fromstring(xml_data)
      total = root.find("opensearch:totalResults", NS)
      return int(total.text) if total is not None else 0

  except Exception as e:
    print(f" Error fetching {category}: {e}")
    return 0

In [7]:
# collect paper counts

results = []

for cat in CATEGORIES:
  count = get_cat_volume(cat)
  results.append({"category": cat, "total_papers": count})
  print(f"{cat}: {count} papers")
  time.sleep(20)

cs.AI: 168181 papers
cs.AR: 7791 papers
cs.CC: 12592 papers
cs.CE: 10072 papers
cs.CG: 7907 papers
cs.CL: 104601 papers
cs.CR: 46546 papers
cs.CV: 185665 papers
cs.CY: 27354 papers
cs.DB: 11369 papers
cs.DC: 27724 papers
cs.DL: 5870 papers
cs.DM: 15372 papers
cs.DS: 28222 papers
cs.ET: 6962 papers
cs.FL: 5866 papers
cs.GL: 229 papers
cs.GT: 14333 papers
cs.HC: 29041 papers
cs.IR: 24351 papers
cs.IT: 53521 papers
cs.LO: 18901 papers
cs.LG: 258190 papers
cs.MA: 11553 papers
cs.MS: 2580 papers
cs.MM: 9230 papers
cs.NE: 17322 papers
cs.NI: 26796 papers
cs.OH: 2306 papers
cs.OS: 1283 papers
cs.PF: 5068 papers
cs.PL: 9537 papers
cs.RO: 50916 papers
cs.SC: 3061 papers
cs.SD: 20214 papers
cs.SE: 25302 papers
cs.SI: 23069 papers


In [8]:
# build dataframe and rank by volume

df = pd.DataFrame(results).sort_values("total_papers",ascending=False).reset_index(drop=True)
df.index +=1
print(df.to_string())

   category  total_papers
1     cs.LG        258190
2     cs.CV        185665
3     cs.AI        168181
4     cs.CL        104601
5     cs.IT         53521
6     cs.RO         50916
7     cs.CR         46546
8     cs.HC         29041
9     cs.DS         28222
10    cs.DC         27724
11    cs.CY         27354
12    cs.NI         26796
13    cs.SE         25302
14    cs.IR         24351
15    cs.SI         23069
16    cs.SD         20214
17    cs.LO         18901
18    cs.NE         17322
19    cs.DM         15372
20    cs.GT         14333
21    cs.CC         12592
22    cs.MA         11553
23    cs.DB         11369
24    cs.CE         10072
25    cs.PL          9537
26    cs.MM          9230
27    cs.CG          7907
28    cs.AR          7791
29    cs.ET          6962
30    cs.DL          5870
31    cs.FL          5866
32    cs.PF          5068
33    cs.SC          3061
34    cs.MS          2580
35    cs.OH          2306
36    cs.OS          1283
37    cs.GL           229


In [9]:
# apply the threshold - 5000 papers

treshold = 5000
selected = df[df["total_papers"] >= treshold].copy()
excluded = df[df["total_papers"] < treshold].copy()

print("Selected papers:")
print(selected.to_string())
if len(excluded)>0:
    print("\n Excluded papers:")
    print(excluded.to_string())
print(f"\n {len(selected)} selected, {len(excluded)} excluded\n")

Selected papers:
   category  total_papers
1     cs.LG        258190
2     cs.CV        185665
3     cs.AI        168181
4     cs.CL        104601
5     cs.IT         53521
6     cs.RO         50916
7     cs.CR         46546
8     cs.HC         29041
9     cs.DS         28222
10    cs.DC         27724
11    cs.CY         27354
12    cs.NI         26796
13    cs.SE         25302
14    cs.IR         24351
15    cs.SI         23069
16    cs.SD         20214
17    cs.LO         18901
18    cs.NE         17322
19    cs.DM         15372
20    cs.GT         14333
21    cs.CC         12592
22    cs.MA         11553
23    cs.DB         11369
24    cs.CE         10072
25    cs.PL          9537
26    cs.MM          9230
27    cs.CG          7907
28    cs.AR          7791
29    cs.ET          6962
30    cs.DL          5870
31    cs.FL          5866
32    cs.PF          5068

 Excluded papers:
   category  total_papers
33    cs.SC          3061
34    cs.MS          2580
35    cs.OH          2306
36

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import os

# create folder
os.makedirs("/content/drive/MyDrive/CM3070_final_project/backend/data", exist_ok=True)

# save files
df.to_csv("/content/drive/MyDrive/CM3070_final_project/backend/data/all_category_volumes.csv", index=True)
selected.to_csv("/content/drive/MyDrive/CM3070_final_project/backend/data/selected_categories.csv", index=False)